<a href="https://colab.research.google.com/github/HARSHITHA24MANVI/AI-Lab-5H/blob/main/1BM24CS114_MACHINE_LEARNING_FEATURE_SELECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install numpy scikit-learn


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

# 2. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Initialize Genetic Algorithm
n_features = X.shape[1]
population_size = 10
generations = 10

np.random.seed(42)
population = np.random.randint(
    0, 2, (population_size, n_features)
)

# 4. Fitness function
def fitness(chromosome):
    selected = np.where(chromosome == 1)[0]

    if len(selected) == 0:
        return 0

    model = RandomForestClassifier(
        n_estimators=50, random_state=42
    )

    model.fit(X_train[:, selected], y_train)
    predictions = model.predict(X_test[:, selected])

    accuracy = accuracy_score(y_test, predictions)

    # Prefer accuracy and fewer features
    score = accuracy - 0.01 * (
        len(selected) / n_features
    )

    return score

# 5. Genetic Algorithm
for generation in range(generations):

    scores = [fitness(ch) for ch in population]

    # Selection: choose the best five chromosomes
    best_indices = np.argsort(scores)[-5:]
    parents = population[best_indices]

    new_population = [parents[-1].copy()]

    while len(new_population) < population_size:

        # Select two parents
        p1, p2 = parents[
            np.random.randint(0, len(parents), 2)
        ]

        # Crossover
        point = np.random.randint(1, n_features)
        child = np.concatenate(
            (p1[:point], p2[point:])
        )

        # Mutation
        for i in range(n_features):
            if np.random.rand() < 0.05:
                child[i] = 1 - child[i]

        new_population.append(child)

    population = np.array(new_population)

# 6. Find the best chromosome
scores = [fitness(ch) for ch in population]
best_chromosome = population[np.argmax(scores)]

selected_features = np.where(
    best_chromosome == 1
)[0]

# 7. Train final classification model
model = RandomForestClassifier(
    n_estimators=50, random_state=42
)

model.fit(X_train[:, selected_features], y_train)

# 8. Predict output
predictions = model.predict(
    X_test[:, selected_features]
)

accuracy = accuracy_score(y_test, predictions)

# 9. Display results
print("Selected Features:")
print(data.feature_names[selected_features])

print("\nTotal Features:", n_features)
print("Selected Features:", len(selected_features))
print("Predicted Class Labels:", predictions)
print("Classification Accuracy:", round(accuracy * 100, 2), "%")



Selected Features:
['mean compactness' 'mean symmetry' 'radius error' 'smoothness error'
 'compactness error' 'symmetry error' 'worst perimeter' 'worst smoothness'
 'worst fractal dimension']

Total Features: 30
Selected Features: 9
Predicted Class Labels: [1 0 0 1 1 0 0 0 1 1 1 0 1 0 1 0 1 1 1 0 0 1 0 1 1 1 1 1 1 0 1 1 1 1 1 1 0
 1 0 1 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 0 1 1 0 0 1 1 1 0 0 1 1 0 0 1 0
 1 1 1 0 1 1 0 1 0 0 0 0 0 0 1 1 1 1 1 1 1 1 0 0 1 0 0 1 0 0 1 1 1 0 1 1 0
 1 1 0]
Classification Accuracy: 100.0 %
